# 04 - Analise Estatistica Inicial da Segmentacao Bruta

Materializa a base analitica da segmentacao bruta a partir do SQLite do projeto, persiste os resultados iniciais da analise estatistica e exibe um resumo por modelo.


In [ ]:
import sys
from pathlib import Path

import pandas as pd

root_dir = Path.cwd()
if not (root_dir / 'src').exists() and (root_dir.parent / 'src').exists():
    root_dir = root_dir.parent

if str(root_dir) not in sys.path:
    sys.path.insert(0, str(root_dir))

from src.analysis import (
    MetricsCollector,
    build_and_persist_analysis_segmentacao_bruta_resumo_execucao,
    build_and_persist_analysis_segmentacao_bruta_resumo_modelo,
    build_and_persist_analysis_segmentacao_bruta_resumo_tag,
)
from src.repositories import (
    AnaliseSegmentacaoBrutaResumoExecucaoRepository,
    AnaliseSegmentacaoBrutaResumoModeloRepository,
    AnaliseSegmentacaoBrutaResumoTagRepository,
)


## Carregamento da base analitica em memoria

Le a avaliacao persistida no SQLite e monta a base analitica por `imagem + modelo + execucao` em memoria.


In [ ]:
collector = MetricsCollector(force_recalculate=False)
df_base = collector.collect_all_metrics()

print(f'Registros na base analitica: {len(df_base)}')
print(f'Imagens: {df_base["nome_arquivo"].nunique()}')
print(f'Modelos: {df_base["modelo"].nunique()}')
print(f'Execucoes: {df_base["execucao"].nunique()}')

df_base.head()


## Resumos estatisticos iniciais persistidos

Calcula as estatisticas descritivas iniciais das metricas brutas e grava as tabelas de resumo por modelo, execucao e tag.


In [ ]:
resumo_modelo_repository = AnaliseSegmentacaoBrutaResumoModeloRepository()
resumo_execucao_repository = AnaliseSegmentacaoBrutaResumoExecucaoRepository()
resumo_tag_repository = AnaliseSegmentacaoBrutaResumoTagRepository()

df_resumo_modelo, _ = build_and_persist_analysis_segmentacao_bruta_resumo_modelo(
    df_base=df_base,
    repository=resumo_modelo_repository,
)
df_resumo_execucao, _ = build_and_persist_analysis_segmentacao_bruta_resumo_execucao(
    df_base=df_base,
    repository=resumo_execucao_repository,
)
df_resumo_tag, _ = build_and_persist_analysis_segmentacao_bruta_resumo_tag(
    df_base=df_base,
    repository=resumo_tag_repository,
)

print(f'Registros no resumo por modelo: {len(df_resumo_modelo)}')
print(f'Registros no resumo por execucao: {len(df_resumo_execucao)}')
print(f'Registros no resumo por tag: {len(df_resumo_tag)}')
df_resumo_modelo


## Leitura orientada dos resultados

Mostra uma visao compacta das medias e medianas por modelo, por execucao e por tag para orientar a etapa seguinte do notebook.


In [ ]:
pd.pivot_table(
    df_resumo_modelo,
    index='modelo',
    columns='metric_name',
    values=['mean', 'median'],
)


In [ ]:
df_resumo_execucao.head()


In [ ]:
df_resumo_tag.head()
